In [ ]:
# read file

import pickle  # type: ignore[reportMissingImports]
import sys
import types
from pathlib import Path  # type: ignore[reportMissingImports]

# 1) Pick the exact file to load
# LiH 8-qubit active-space case generated in Pauli_Ham
file_name = "LiH_active8_scan_bond_1.596_active_1-2-4-5_q8_of.pkl"

# 2) Resolve Pauli_Ham folder based on runtime
try:
    import google.colab  # type: ignore[reportMissingImports]
    in_colab = True
except ImportError:
    in_colab = False

if in_colab:
    # Keep your original Google Drive path behavior in Colab
    save_folder = Path('/content/drive/My Drive/Quantum_chemistry/pauli_Ham')
else:
    # Local fallback: prefer ./Pauli_Ham, then ../Pauli_Ham
    cwd = Path.cwd()
    local_candidates = [cwd / 'Pauli_Ham', cwd.parent / 'Pauli_Ham']
    save_folder = next((p for p in local_candidates if p.exists()), local_candidates[0])

file_path = save_folder / file_name


class PauliSumOpShim:
    """Compatibility shim for loading old qiskit.opflow pickles."""

    def __init__(self, *args, **kwargs):
        pass


def install_qiskit_opflow_shim() -> None:
    """Provide legacy qiskit.opflow module path for pickle loading."""
    module_name = 'qiskit.opflow.primitive_ops.pauli_sum_op'

    # Rebind every run so class identity remains consistent across reruns.
    opflow_module = sys.modules.get('qiskit.opflow', types.ModuleType('qiskit.opflow'))
    primitive_ops_module = sys.modules.get(
        'qiskit.opflow.primitive_ops',
        types.ModuleType('qiskit.opflow.primitive_ops')
    )
    pauli_sum_op_module = sys.modules.get(module_name, types.ModuleType(module_name))
    pauli_sum_op_module.PauliSumOp = PauliSumOpShim

    sys.modules['qiskit.opflow'] = opflow_module
    sys.modules['qiskit.opflow.primitive_ops'] = primitive_ops_module
    sys.modules[module_name] = pauli_sum_op_module


def to_sparse_pauli_op(operator):
    """Normalize loaded Hamiltonians to qiskit's SparsePauliOp."""
    from qiskit.quantum_info import SparsePauliOp  # type: ignore[reportMissingImports]

    # Legacy qiskit.opflow object
    if hasattr(operator, '_primitive'):
        operator = operator._primitive

    # Already in the expected format
    if isinstance(operator, SparsePauliOp):
        return operator

    # OpenFermion QubitOperator -> SparsePauliOp
    if hasattr(operator, 'terms'):
        n_qubits = 0
        for term in operator.terms:
            for qubit_idx, _ in term:
                n_qubits = max(n_qubits, qubit_idx + 1)

        pauli_terms = []
        for term, coeff in operator.terms.items():
            label = ['I'] * n_qubits
            for qubit_idx, pauli in term:
                # Qiskit labels are ordered q_(n-1) ... q_0
                label[n_qubits - 1 - qubit_idx] = pauli
            pauli_terms.append((''.join(label), complex(coeff)))

        return SparsePauliOp.from_list(pauli_terms).simplify()

    raise TypeError(f'Unsupported Hamiltonian type: {type(operator)}')


# 3) Load the object
if file_path.exists():
    install_qiskit_opflow_shim()

    with file_path.open('rb') as file:
        H_qubit_loaded = pickle.load(file)

    H_qubit_loaded = to_sparse_pauli_op(H_qubit_loaded)

    print('--- Hamiltonian Loaded Successfully ---')
    print(f"Runtime: {'Google Colab' if in_colab else 'Local'}")
    print(f'Folder: {save_folder}')
    print(f'File: {file_name}')
    print(f'Number of qubits: {H_qubit_loaded.num_qubits}')
    print(f'Number of Pauli terms: {len(H_qubit_loaded)}')
    print(f'Object type: {type(H_qubit_loaded)}')
else:
    print(f'Error: File not found: {file_path}')
    if save_folder.exists():
        pkl_files = sorted(p.name for p in save_folder.glob('*.pkl'))
        print('Available .pkl files in folder:')
        for name in pkl_files:
            print(f'  - {name}')
    else:
        print('Pauli_Ham folder was not found in expected locations.')


In [ ]:
from qiskit import QuantumCircuit  # type: ignore[reportMissingImports]
from qiskit.circuit import ParameterVector  # type: ignore[reportMissingImports]

def givens_block_fermionic(qc, theta, q1, q2):
    """
    Implements: exp[ i * theta * (c^\dagger_1 c_2 + c^\dagger_2 c_1) ]
    Assumes q1 and q2 are adjacent qubits.
    """
    # Multiply by -1 to account for Qiskit's RXX/RYY definitions
    # RXX(a) * RYY(a) = exp(-i * a/2 * (XX+YY)). We want exp(i * theta/2 * (XX+YY)).

    qc.rxx(-theta, q1, q2)
    qc.ryy(-theta, q1, q2)

def onsite_block(qc, phi, q_alpha, q_beta):
    """ZZ interaction (O block) between opposite spin orbitals."""
    qc.rzz(-phi/2, q_alpha, q_beta)
    qc.rz(phi/2, q_alpha)
    qc.rz(phi/2, q_beta)

def prepare_ansatz(num_spatial_orbitals, num_layers=1):
    num_qubits = 2 * num_spatial_orbitals
    qc = QuantumCircuit(num_qubits)

    # Parameter Count:
    # (num_spatial//2) + ((num_spatial-1)//2) + num_spatial
    params_per_layer = (num_spatial_orbitals // 2) + ((num_spatial_orbitals - 1) // 2) + num_spatial_orbitals
    params = ParameterVector('θ', length=num_layers * params_per_layer)
    p_idx = 0

    # --- STEP 1: INITIAL STATE ---
    for i in (0, 4): ##### ADJUST THIS FOR INITIAL STATE STATUS
        qc.x(i)
    qc.barrier()

    for layer in range(num_layers):
        # --- STEP 2: GIVENS ROTATION LAYER G (Even-Odd) ---
        # Share theta between alpha and beta sectors
        for i in range(0, num_spatial_orbitals - 1, 2):
            # Alpha sector
            givens_block_fermionic(qc, params[p_idx], i, i + 1)
            # Beta sector (offset by num_spatial_orbitals)
            givens_block_fermionic(qc, params[p_idx], i + num_spatial_orbitals, i + 1 + num_spatial_orbitals)
            p_idx += 1
        qc.barrier()

        # --- STEP 3: GIVENS ROTATION LAYER G' (Odd-Even) ---
        # Share theta between alpha and beta sectors
        for i in range(1, num_spatial_orbitals - 1, 2):
            # Alpha sector
            givens_block_fermionic(qc, params[p_idx], i, i + 1)
            # Beta sector
            givens_block_fermionic(qc, params[p_idx], i + num_spatial_orbitals, i + 1 + num_spatial_orbitals)
            p_idx += 1
        qc.barrier()

        # --- STEP 4: ON-SITE POTENTIAL (O gates) ---
        # Connects qubit i (alpha) with qubit i + num_spatial_orbitals (beta)
        for i in range(num_spatial_orbitals):
            onsite_block(qc, params[p_idx], i, i + num_spatial_orbitals)
            p_idx += 1

        # Keep the layers visually distinct
        qc.barrier()

    return qc

# --- Test Case ---
ansatz = prepare_ansatz(num_spatial_orbitals=4, num_layers=1)
print(f"Number of parameters: {ansatz.num_parameters}")

# Draw the circuit
ansatz.draw('mpl')


In [ ]:
import numpy as np  # type: ignore[reportMissingImports]
from qiskit.primitives import StatevectorEstimator as Estimator  # type: ignore[reportMissingImports]

# 1. Setup the Estimator
estimator = Estimator()

def compute_vqe_energy(params, ansatz, hamiltonian):
    """
    Core function to measure the energy of the ansatz
    for a specific set of parameters.
    """
    # Create the Primitive Unified Bloc (PUB)
    pub = (ansatz, hamiltonian, params)

    # Run the noiseless estimator
    job = estimator.run([pub])
    result = job.result()

    # Extract the expectation value
    energy = result[0].data.evs
    return float(energy)



num_spatial_orbitals = 4  # Reverted to 4, as H4 is an 8-qubit case
#### NUMBER OF LAYERS CHANGE IT HERE
ansatz = prepare_ansatz(num_spatial_orbitals, num_layers=8)
print(f"Number of parameters: {ansatz.num_parameters}")
hamiltonian = H_qubit_loaded

# Let's test two distinct points:
# Point A: All zeros (No hopping, should represent the initial product state)
params_zeros = np.zeros(ansatz.num_parameters)
energy_zero = compute_vqe_energy(params_zeros, ansatz, hamiltonian)

# Point B:
params_random = 0*np.ones(ansatz.num_parameters)

energy_random = compute_vqe_energy(params_random, ansatz, hamiltonian)

print(f"--- Energy Check ---")
print(f"Parameters (Zeros)  Energy: {energy_zero:.12f}")
print(f"Parameters (Random) Energy: {energy_random:.12f}")


In [ ]:
import numpy as np  # type: ignore[reportMissingImports]
from qiskit.quantum_info import Statevector, state_fidelity  # type: ignore[reportMissingImports]

def get_exact_ground_state(hamiltonian):
    """
    Diagonalizes the Hamiltonian to find the exact ground state energy and statevector.
    """
    # 1. Convert the Qiskit operator to a dense numpy matrix
    ham_matrix = hamiltonian.to_matrix()

    # 2. Diagonalize the Hamiltonian matrix (eigh is for Hermitian matrices)
    eigenvalues, eigenvectors = np.linalg.eigh(ham_matrix)

    # 3. Extract the lowest energy and corresponding eigenvector
    exact_energy = eigenvalues[0]
    exact_gs_array = eigenvectors[:, 0]

    # 4. Convert it to a Qiskit Statevector
    exact_gs_statevector = Statevector(exact_gs_array)

    return exact_energy, exact_gs_statevector


def get_fidelity(state_1, state_2):
    """
    Calculates the fidelity between two quantum states.
    """
    # Compute the squared overlap |<state_1 | state_2>|^2
    fidelity = state_fidelity(state_1, state_2)
    return fidelity


# --- Usage ---

# 1. Get the exact ground state and energy
exact_energy, exact_gs_statevector = get_exact_ground_state(hamiltonian)


In [ ]:
# import numpy as np
# from types import SimpleNamespace
# from qiskit_algorithms.gradients import ParamShiftEstimatorGradient

# class VQETracker:
#     def __init__(self, ansatz, hamiltonian, estimator):
#         self.ansatz = ansatz
#         self.hamiltonian = hamiltonian
#         self.estimator = estimator
#         self.gradient = ParamShiftEstimatorGradient(self.estimator)
#         self.energy_history = []
#         self.grad_norm_history = []
#         self.last_grad_norm = 0.0
#         self.last_energy = 0.0
#         self.total_measurements = 0
#         self.energy_meas_this_iter = 0
#         self.gradient_meas_this_iter = 0

#     def objective(self, params):
#         pub = (self.ansatz, self.hamiltonian, params)
#         job = self.estimator.run([pub])
#         result = job.result()
#         energy = float(result[0].data.evs)

#         self.energy_meas_this_iter += 1
#         self.total_measurements += 1
#         self.last_energy = energy
#         return energy

#     def gradient_func(self, params):
#         gradient_job = self.gradient.run([self.ansatz], [self.hamiltonian], [params])
#         gradient_result = gradient_job.result()
#         grad = np.asarray(gradient_result.gradients[0], dtype=float)

#         num_params = len(params)
#         self.gradient_meas_this_iter += 2 * num_params
#         self.total_measurements += 2 * num_params
#         self.last_grad_norm = np.linalg.norm(grad)
#         return grad

#     def log_iteration(self):
#         self.energy_history.append(self.last_energy)
#         self.grad_norm_history.append(self.last_grad_norm)
#         iteration = len(self.energy_history)
#         print(
#             f"Iteration {iteration:3d} | Energy: {self.last_energy:.12f} | "
#             f"Grad Norm: {self.last_grad_norm:.12f} | Grad Meas: {self.gradient_meas_this_iter} | "
#             f"Energy Meas: {self.energy_meas_this_iter} | Total: {self.total_measurements}"
#         )
#         self.energy_meas_this_iter = 0
#         self.gradient_meas_this_iter = 0


# def run_adam(tracker, x0, maxiter=1000, lr=1, beta1=0.9, beta2=0.999, eps=1e-8, grad_tol=1e-8):
#     params = np.array(x0, dtype=float)
#     m = np.zeros_like(params)
#     v = np.zeros_like(params)

#     for t in range(1, maxiter + 1):
#         grad = tracker.gradient_func(params)
#         m = beta1 * m + (1.0 - beta1) * grad
#         v = beta2 * v + (1.0 - beta2) * (grad ** 2)
#         m_hat = m / (1.0 - beta1**t)
#         v_hat = v / (1.0 - beta2**t)
#         params -= lr * m_hat / (np.sqrt(v_hat) + eps)

#         energy = tracker.objective(params)
#         tracker.log_iteration()

#         if tracker.last_grad_norm < grad_tol:
#             break

#     return SimpleNamespace(
#         x=params,
#         fun=energy,
#         success=tracker.last_grad_norm < grad_tol,
#         nit=len(tracker.energy_history),
#     )


# # 1. Initialize the tracker
# tracker = VQETracker(ansatz, hamiltonian, estimator)

# # 2. Define the initial parameters
# #initial_params = np.random.uniform(-np.pi / 4, np.pi / 4, ansatz.num_parameters)
# initial_params = np.random.uniform(-0.01, 0.01, ansatz.num_parameters)

# # 3. Run Adam
# print("\n--- Starting Adam Optimization ---")
# result = run_adam(
#     tracker,
#     initial_params,
#     maxiter=100,
#     lr=0.01,
#     beta1=0.9,
#     beta2=0.999,
#     eps=1e-8,
#     grad_tol=1e-8,
# )

# # 4. Display the results
# print("\n--- Optimization Complete ---")
# print(f"Success: {result.success}")
# print(f"Number of Iterations: {result.nit}")
# print(f"Final Total Measurements: {tracker.total_measurements}")
# print(f"Final Optimized Energy: {result.fun:.12f}")
# print(f"Final Gradient Norm: {tracker.last_grad_norm:.12f}")


In [ ]:
import numpy as np  # type: ignore[reportMissingImports]
from scipy.optimize import minimize  # type: ignore[reportMissingImports]
from qiskit_algorithms.gradients import ParamShiftEstimatorGradient  # type: ignore[reportMissingImports]

# Set the noise level
noise_level = 0

# Counters for measurements
total_measurements_lbfgs = 0
energy_meas_this_iter = 0
gradient_meas_this_iter = 0
iteration_count = 0
latest_noisy_energy = np.nan

def noisy_objective(params):
    global total_measurements_lbfgs, energy_meas_this_iter, latest_noisy_energy
    """Evaluates energy and adds simulated Gaussian noise."""
    pub = (ansatz, hamiltonian, params)
    exact_energy = float(estimator.run([pub]).result()[0].data.evs)
    # Inject random noise
    noisy_energy = exact_energy + np.random.normal(0, noise_level)
    latest_noisy_energy = noisy_energy
    total_measurements_lbfgs += 1
    energy_meas_this_iter += 1
    return noisy_energy

# Initialize Qiskit's Parameter-Shift Gradient Calculator
gradient_calculator = ParamShiftEstimatorGradient(estimator)

def noisy_gradient(params):
    global total_measurements_lbfgs, gradient_meas_this_iter
    """Evaluates exact gradient via parameter-shift and adds noise."""
    gradient_job = gradient_calculator.run([ansatz], [hamiltonian], [params])
    exact_grad = gradient_job.result().gradients[0]
    # Inject random noise into the gradient
    noisy_grad = exact_grad + np.random.normal(0, noise_level, size=len(exact_grad))
    num_params = len(params)
    total_measurements_lbfgs += 2 * num_params
    gradient_meas_this_iter += 2 * num_params
    return noisy_grad

def callback(intermediate_result):
    global iteration_count, energy_meas_this_iter, gradient_meas_this_iter, total_measurements_lbfgs, latest_noisy_energy
    iteration_count += 1
    print(f"Iteration {iteration_count:3d} | Energy: {latest_noisy_energy:.10f} | Grad Meas: {gradient_meas_this_iter} | Energy Meas: {energy_meas_this_iter} | Cumulative: {total_measurements_lbfgs}")
    # Reset counters for the next iteration
    energy_meas_this_iter = 0
    gradient_meas_this_iter = 0

print(f"--- Testing L-BFGS-B with Noisy Parameter-Shift Gradient (std = {noise_level}) ---")

# We start from random parameters again
#initial_params_noisy = np.random.uniform(-np.pi/4, np.pi/4, ansatz.num_parameters)
initial_params_noisy = np.ones(ansatz.num_parameters)*0.5

# Run L-BFGS-B on the noisy objective, now providing the noisy parameter-shift gradient
result_noisy = minimize(
    fun=noisy_objective,
    x0=initial_params_noisy,
    jac=noisy_gradient,  # Use parameter-shift rule here
    method='L-BFGS-B',
    callback=callback,
    options={'maxiter': 100, 'ftol': 1e-8}
)

# Evaluate the TRUE (noiseless) energy of the final parameters found by the noisy optimizer
final_pub = (ansatz, hamiltonian, result_noisy.x)
true_energy_of_noisy_opt = float(estimator.run([final_pub]).result()[0].data.evs)

print("\n--- Results ---")
print(f"Optimization Success: {result_noisy.success} (Often false due to noise)")
print(f"Iterations: {result_noisy.nit}")
print(f"Function Evaluations: {result_noisy.nfev}")
print(f"Total Measurements: {total_measurements_lbfgs}")
print(f"Noisy Energy reported by optimizer:   {result_noisy.fun:.6f}")
print(f"True (Noiseless) Energy of result:  {true_energy_of_noisy_opt:.6f}")
print(f"Exact Ground State Energy:            {exact_energy.real:.6f}")
print(f"Absolute Error vs Exact:              {abs(true_energy_of_noisy_opt - exact_energy.real):.6f}")


In [ ]:
# from qiskit_algorithms.optimizers import GradientDescent
# import time

# print(f"--- Testing SGD with Artificial Noise (std = {noise_level}) ---")

# iteration_count_sgd = 0

# def sgd_callback(nfevs, x, fx, stepsize, *args):
#     global iteration_count_sgd, energy_meas_this_iter, gradient_meas_this_iter, total_measurements_lbfgs
#     iteration_count_sgd += 1
#     print(f"Iteration {iteration_count_sgd:3d} | Energy: {fx:.6f} | Grad Meas: {gradient_meas_this_iter} | Energy Meas: {energy_meas_this_iter} | Cumulative: {total_measurements_lbfgs}")
#     # Reset counters for the next iteration
#     energy_meas_this_iter = 0
#     gradient_meas_this_iter = 0

# # Initialize Gradient Descent (learning_rate can be tuned)
# # SGD takes longer because it was set to run for 300 iterations without early stopping.
# # L-BFGS-B stopped early (e.g., 30 iterations) due to its advanced convergence criteria.
# sgd = GradientDescent(maxiter=100, learning_rate=0.5, callback=sgd_callback)

# # Reset iteration counters before starting SGD
# energy_meas_this_iter = 0
# gradient_meas_this_iter = 0
# total_measurements_lbfgs = 0

# start_time = time.time()
# # Run SGD optimization using the noisy objective and noisy gradient
# sgd_result = sgd.minimize(fun=noisy_objective, x0=initial_params_noisy, jac=noisy_gradient)
# end_time = time.time()

# # Evaluate the TRUE (noiseless) energy of the final parameters found by SGD
# final_pub_sgd = (ansatz, hamiltonian, sgd_result.x)
# true_energy_of_sgd_opt = float(estimator.run([final_pub_sgd]).result()[0].data.evs)

# print("\n--- SGD Results ---")
# print(f"Iterations: {sgd_result.nit}")
# print(f"Function Evaluations: {sgd_result.nfev}")
# print(f"Time elapsed: {end_time - start_time:.2f} seconds")
# print(f"Noisy Energy reported by optimizer:   {sgd_result.fun:.6f}")
# print(f"True (Noiseless) Energy of result:  {true_energy_of_sgd_opt:.6f}")
# print(f"Exact Ground State Energy:            {exact_energy.real:.6f}")
# print(f"Absolute Error vs Exact:              {abs(true_energy_of_sgd_opt - exact_energy.real):.6f}")

# print("\n--- Comparison against L-BFGS-B ---")
# print(f"L-BFGS-B True Energy: {true_energy_of_noisy_opt:.6f}")
# print(f"SGD True Energy:      {true_energy_of_sgd_opt:.6f}")
# if true_energy_of_sgd_opt < true_energy_of_noisy_opt:
#     print("\nResult: SGD achieved a lower (better) energy under noise!")
# else:
#     print("\nResult: L-BFGS-B achieved a lower energy here.")


In [ ]:
from qiskit.quantum_info import Statevector  # type: ignore[reportMissingImports]

def get_wavefunction(params, ansatz):
    # 1. Bind the numerical parameters to the circuit
    bound_ansatz = ansatz.assign_parameters(params)

    # 2. Compute the statevector
    statevector = Statevector.from_instruction(bound_ansatz)

    return statevector

# Usage:
wavefunction = get_wavefunction(result_noisy.x, ansatz)

# To see the amplitudes/components:
print(len(wavefunction))


In [ ]:
import numpy as np  # type: ignore[reportMissingImports]
from qiskit.quantum_info import Statevector, state_fidelity  # type: ignore[reportMissingImports]

def get_exact_ground_state(hamiltonian):
    """
    Diagonalizes the Hamiltonian to find the exact ground state energy and statevector.
    """
    # 1. Convert the Qiskit operator to a dense numpy matrix
    ham_matrix = hamiltonian.to_matrix()

    # 2. Diagonalize the Hamiltonian matrix (eigh is for Hermitian matrices)
    eigenvalues, eigenvectors = np.linalg.eigh(ham_matrix)

    # 3. Extract the lowest energy and corresponding eigenvector
    exact_energy = eigenvalues[0]
    exact_gs_array = eigenvectors[:, 0]

    # 4. Convert it to a Qiskit Statevector
    exact_gs_statevector = Statevector(exact_gs_array)

    return exact_energy, exact_gs_statevector


def get_fidelity(state_1, state_2):
    """
    Calculates the fidelity between two quantum states.
    """
    # Compute the squared overlap |<state_1 | state_2>|^2
    fidelity = state_fidelity(state_1, state_2)
    return fidelity


# --- Usage ---

# 1. Get the exact ground state and energy
exact_energy, exact_gs_statevector = get_exact_ground_state(hamiltonian)

# 2. Calculate the fidelity between your VQE wavefunction and the exact one
# (Assuming 'wavefunction' is already prepared using your get_wavefunction code)
fidelity = get_fidelity(wavefunction, exact_gs_statevector)

# --- Display Results ---
print(f"--- Results ---")
print(f"VQE Final Energy:    {result_noisy.fun:.12f}")
print(f"Exact Ground Energy: {exact_energy.real:.12f}")
print(f"Energy Difference:   {abs(result_noisy.fun - exact_energy.real):.12e}")
print(f"State Fidelity:      {fidelity:.12f}")


In [ ]:
# 2.886494100260e-03 exact


In [ ]:
# array([ 0.62769563,  0.72445584,  0.88167509,  1.21377448,  1.40410528,
#         1.54272135,  1.65876094,  0.68703915,  0.53914478,  0.65488307,
#         0.38242015,  1.21990629, -0.37586606, -0.55407695,  1.14433297,
#         0.93390869,  1.06005155,  1.13875818,  0.31887654,  1.95665943,
#         0.92840525,  0.96902297,  0.62448309,  0.85426258,  3.07549546,
#         1.06236478,  1.83665633, -0.07343972,  1.05494565,  0.51913651,
#         0.8990249 , -0.7786491 ,  0.20801007,  1.33467266,  1.50009365,
#         0.75667649,  1.43658698,  1.15788765,  0.93250988,  0.21547095,
#         1.14415573, -0.34546883,  1.06523003,  1.45429555,  0.63474669,
#         1.32076148,  0.51153084,  1.32344213,  0.64436686,  1.11864704,
#         0.46516075,  1.52517135,  1.25912853,  0.6848312 , -0.26316329,
#        -0.07124165] exact with error --- Results ---
# VQE Final Energy:    -3.061944645714
# Exact Ground Energy: -3.062277999937
# Energy Difference:   3.333542222768e-04
# State Fidelity:      0.998270269221


In [ ]:
import numpy as np  # type: ignore[reportMissingImports]

# Define the array of fine-grained noise levels.
# These specific noise levels are chosen to systematically probe the optimizer limits
# without completely destroying the gradient signal, allowing us to find the
# max tolerable noise at different stages of the optimization.
noise_levels = [0.1, 0.08, 0.05, 0.03, 0.02, 0.01, 0.005,0.002,0.001,0.0005,0.0001]

# Ensure ansatz is defined
num_spatial_orbitals = 4
ansatz = prepare_ansatz(num_spatial_orbitals, num_layers=8)

# Define a fixed set of initial parameters for the ansatz.
# This ensures all sweeps in the next step start from the exact same point for a fair comparison.
np.random.seed(42)
fixed_initial_params = np.random.uniform(-np.pi/4, np.pi/4, ansatz.num_parameters)

print("Noise levels to test:", noise_levels)
print("Fixed initial parameters shape:", fixed_initial_params.shape)


# Task
Find the maximum tolerable noise thresholds for optimizing the VQE by setting up a fine-grained noise array (e.g., [0.1, 0.08, 0.05, 0.03, 0.02, 0.01, 0.005]) and running L-BFGS-B optimization sweeps from the exact same initial parameters. Track and plot the true (noiseless) energy against the iteration number for each noise level in a single plot. Analyze the plotted data to find the empirical maximum tolerable noise at different iteration milestones (e.g., iterations 5, 10, 20), and use these findings to construct a concrete, piece-wise dynamic noise schedule. Include explanatory comments for the noise bounds.

## Execute Noise Sweeps

### Subtask:
Run the L-BFGS-B optimization independently for each defined noise level starting from the fixed initial parameters, and track the true (noiseless) energy at each iteration.


**Reasoning**:
I am creating a code cell to run the L-BFGS-B optimization for each noise level, tracking the true noiseless energy history using a callback.



In [ ]:
import numpy as np  # type: ignore[reportMissingImports]
from scipy.optimize import minimize  # type: ignore[reportMissingImports]
from qiskit_algorithms.gradients import ParamShiftEstimatorGradient  # type: ignore[reportMissingImports]

energy_histories = {}
gradient_calculator = ParamShiftEstimatorGradient(estimator)

for current_noise in noise_levels:
    print(f"Starting sweep for noise level: {current_noise}")
    current_history = []

    def noisy_objective_sweep(params):
        pub = (ansatz, hamiltonian, params)
        exact_energy = float(estimator.run([pub]).result()[0].data.evs)
        return exact_energy + np.random.normal(0, current_noise)

    def noisy_gradient_sweep(params):
        gradient_job = gradient_calculator.run([ansatz], [hamiltonian], [params])
        exact_grad = gradient_job.result().gradients[0]
        return exact_grad + np.random.normal(0, current_noise, size=len(exact_grad))

    def callback_sweep(intermediate_result):
        # Handle both SciPy versions (array vs OptimizeResult)
        xk = getattr(intermediate_result, 'x', intermediate_result)

        # We calculate the true energy (noiseless) for the current parameter vector
        pub = (ansatz, hamiltonian, xk)
        true_energy = float(estimator.run([pub]).result()[0].data.evs)
        current_history.append(true_energy)

    # Run L-BFGS-B optimizer
    result = minimize(
        fun=noisy_objective_sweep,
        x0=fixed_initial_params,
        jac=noisy_gradient_sweep,
        method='L-BFGS-B',
        callback=callback_sweep,
        options={'maxiter': 50}
    )

    energy_histories[current_noise] = current_history
    final_energy = current_history[-1] if current_history else float('nan')
    print(f"Completed sweep for noise {current_noise}. Final true energy: {final_energy:.6f}\n")


# Task
Find the maximum tolerable noise thresholds for optimizing the VQE by setting up a fine-grained noise array and running sweeps from the exact same initial parameters. Plot the true noiseless energy against the iteration number for each noise level. Analyze the plotted data to find empirical thresholds at iteration milestones (e.g., 5, 10, 20), and construct a piece-wise dynamic noise schedule function based on these empirical thresholds. Finally, provide a summary of the empirical findings and the finalized dynamic noise schedule.

## Plot Energy Histories

### Subtask:
Generate a plot visualizing the true (noiseless) energy against the iteration number for all tested noise levels.


**Reasoning**:
Create a plot to visualize how the true (noiseless) energy evolves over iterations for each noise level, helping us identify the threshold where noise prevents convergence to the exact ground state energy.

**Reasoning**:
Generate the code block to plot the optimization trajectories for all noise levels.



In [ ]:
import matplotlib.pyplot as plt  # type: ignore[reportMissingImports]

plt.figure(figsize=(10, 6))

for noise, history in energy_histories.items():
    plt.plot(range(1, len(history) + 1), history, marker='o', label=f'Noise: {noise}')

plt.axhline(y=exact_energy.real, color='r', linestyle='--', label='Exact Ground State')

plt.xlabel('Iteration')
plt.ylabel('True (Noiseless) Energy')
plt.title('VQE Optimization Trajectories under Various Noise Levels')
plt.legend()
plt.grid(True)
plt.show()


**Reasoning**:
I am creating a Python function to establish a dynamic noise schedule based on the empirical observations from the energy histories plot.



In [ ]:
def dynamic_noise_schedule(iteration):
    """
    Returns the maximum tolerable noise level for a given iteration based on empirical findings.

    - Iterations 1-5: High noise tolerance. The optimizer can handle larger noise (e.g., 0.1) while still decreasing energy significantly from a random start.
    - Iterations 6-10: Moderate noise tolerance. As the optimizer approaches a local minimum, it requires smaller noise (e.g., 0.05) to continue descending.
    - Iterations 11-20: Low noise tolerance. Finer steps require even less noise (e.g., 0.02) to avoid bouncing around the minimum.
    - Iterations >20: Very low noise tolerance. To reach the exact ground state energy, the noise must be minimal (e.g., 0.005).
    """
    if iteration <= 5:
        return 0.08
    elif iteration <= 10:
        return 0.05
    elif iteration <= 20:
        return 0.005
    else:
        return 0.002

# Test the function
print("Iteration 2 noise bound:", dynamic_noise_schedule(2))
print("Iteration 8 noise bound:", dynamic_noise_schedule(8))
print("Iteration 15 noise bound:", dynamic_noise_schedule(15))
print("Iteration 25 noise bound:", dynamic_noise_schedule(25))


In [ ]:
import numpy as np  # type: ignore[reportMissingImports]
import matplotlib.pyplot as plt  # type: ignore[reportMissingImports]

def get_dynamic_noise(iteration):
    """
    Dynamic noise schedule based on empirical thresholds.
    Starts high and decreases to allow convergence.
    """
    if iteration <= 5:
        return 0.08  # High noise tolerance initially
    elif iteration <= 10:
        return 0.05  # Moderate tolerance
    elif iteration <= 20:
        return 0.01  # Low tolerance for fine-tuning
    elif iteration <= 35:
        return 0.005 # Very low tolerance
    else:
        return 0.001 # Minimal noise near convergence

# Visualize the dynamic noise schedule
iterations = np.arange(1, 51)
noise_vals = [get_dynamic_noise(i) for i in iterations]

plt.figure(figsize=(8, 4))
plt.step(iterations, noise_vals, where='post', color='blue', linewidth=2)
plt.xlabel('Iteration')
plt.ylabel('Tolerable Noise Level (std)')
plt.title('Dynamic Noise Schedule for L-BFGS-B Optimization')
plt.grid(True)
plt.show()


In [ ]:
import numpy as np  # type: ignore[reportMissingImports]
from qiskit_algorithms.gradients import ParamShiftEstimatorGradient  # type: ignore[reportMissingImports]

# Trackers
total_measurements_dynamic = 0
dynamic_energy_history = []

print("--- Starting Custom Fixed-Iteration Optimization with Dynamic Noise Schedule ---")

# Configuration
num_fixed_iterations = 50
learning_rate = 0.5
rejection_threshold = 0.0  # Set to 0.0 to reject ANY step that increases the energy

# Initialize
params = fixed_initial_params.copy()
dynamic_gradient_calculator = ParamShiftEstimatorGradient(estimator)

def evaluate_dynamic_noisy_energy(p, noise):
    global total_measurements_dynamic
    pub = (ansatz, hamiltonian, p)
    exact_e = float(estimator.run([pub]).result()[0].data.evs)
    total_measurements_dynamic += 1
    return exact_e + np.random.normal(0, noise)

def evaluate_dynamic_noisy_gradient(p, noise):
    global total_measurements_dynamic
    grad_job = dynamic_gradient_calculator.run([ansatz], [hamiltonian], [p])
    exact_g = grad_job.result().gradients[0]
    total_measurements_dynamic += 2 * len(p)
    return exact_g + np.random.normal(0, noise, size=len(exact_g))

# Initial evaluation
current_noise = get_dynamic_noise(0)
current_energy = evaluate_dynamic_noisy_energy(params, current_noise)

for i in range(1, num_fixed_iterations + 1):
    # Get dynamic noise for this iteration
    current_noise = get_dynamic_noise(i)

    # Calculate noisy gradient
    noisy_grad = evaluate_dynamic_noisy_gradient(params, current_noise)

    # Propose new parameters
    proposed_params = params - learning_rate * noisy_grad

    # Evaluate energy at proposed parameters
    proposed_energy = evaluate_dynamic_noisy_energy(proposed_params, current_noise)

    # Rejection logic
    if proposed_energy - current_energy > rejection_threshold:
        print(f"Iteration {i:3d} | Noise Level: {current_noise:.5f} | Step Rejected! (Energy spiked from {current_energy:.4f} to {proposed_energy:.4f})")
    else:
        # Accept step
        params = proposed_params
        current_energy = proposed_energy
        print(f"Iteration {i:3d} | Noise Level: {current_noise:.5f} | Step Accepted | Noisy Energy: {current_energy:.6f}")

    # Track true energy for the current params (accepted or not)
    pub = (ansatz, hamiltonian, params)
    true_energy = float(estimator.run([pub]).result()[0].data.evs)
    dynamic_energy_history.append(true_energy)

print("\n--- Optimization Complete ---")
print(f"Total Measurements: {total_measurements_dynamic}")
print(f"Final True Energy: {dynamic_energy_history[-1]:.6f}")
print(f"Exact Ground State Energy: {exact_energy.real:.6f}")
print(f"Absolute Error: {abs(dynamic_energy_history[-1] - exact_energy.real):.6f}")
